# Verify Conv-VAE-Neo

Load a trained internal VAE through the normal sensor-processing factory and inspect its reconstructions and latent representation.

In [ ]:
import json
import math
import pathlib
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo import ConvVAENeoLoss, DemonstrationImageDataset
from sensorprocessing.sp_factory import create_sp

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
experiment = "sensorprocessing_conv_vae_neo"
run = "sp_vae_neo_128_256px"
sample_count = 12

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment(experiment)
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
sp = create_sp(exp)
dataset = DemonstrationImageDataset(exp, "validation_data")
loader = DataLoader(dataset, batch_size=min(sample_count, len(dataset)), shuffle=False)
images = next(iter(loader)).to(Config().runtime["device"])

## Reconstruction and latent metrics

In [ ]:
with torch.no_grad():
    output = sp.enc(images)
reconstructions, mu, logvar = output
components = ConvVAENeoLoss(exp).components(output, images)
mse = float(torch.mean((reconstructions - images) ** 2))
psnr = float("inf") if mse == 0 else 10.0 * math.log10(1.0 / mse)
runtime_latent = sp.process(images[:1])
if runtime_latent.shape != (exp["latent_size"],):
    raise ValueError(f"Unexpected runtime latent shape: {runtime_latent.shape}")
if not np.all(np.isfinite(runtime_latent)):
    raise FloatingPointError("Runtime latent contains non-finite values")
metrics = {
    "loss": float(components["loss"]),
    "reconstruction_loss": float(components["reconstruction"]),
    "kl_loss": float(components["kl"]),
    "mse": mse,
    "psnr": psnr,
    "mu_mean": float(mu.mean()),
    "mu_std": float(mu.std()),
    "logvar_mean": float(logvar.mean()),
    "logvar_std": float(logvar.std()),
}
print(json.dumps(metrics, indent=2))
metrics_path = pathlib.Path(exp.data_dir(), "verification_metrics.json")
with metrics_path.open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
    handle.write("\n")

In [ ]:
originals = images.cpu()
reconstructed = reconstructions.cpu()
count = min(sample_count, originals.size(0))
fig, axes = plt.subplots(2, count, figsize=(3 * count, 6), squeeze=False)
for index in range(count):
    axes[0, index].imshow(originals[index].permute(1, 2, 0))
    axes[0, index].axis("off")
    axes[1, index].imshow(reconstructed[index].permute(1, 2, 0))
    axes[1, index].axis("off")
axes[0, 0].set_title("Original")
axes[1, 0].set_title("Reconstruction")
fig.tight_layout()
figure_path = pathlib.Path(exp.data_dir(), "verification_reconstructions.png")
fig.savefig(figure_path, bbox_inches="tight")
plt.show()

## Samples from the latent prior

In [ ]:
with torch.no_grad():
    samples = sp.enc.sample(16).cpu()
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for image, axis in zip(samples, axes.flat):
    axis.imshow(image.permute(1, 2, 0))
    axis.axis("off")
fig.tight_layout()
sample_path = pathlib.Path(exp.data_dir(), "verification_prior_samples.png")
fig.savefig(sample_path, bbox_inches="tight")
plt.show()